In [1]:
%load_ext autoreload
%autoreload 2

In [45]:
from pathlib import Path
import sys
import os
from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "utils").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / ".env", override=True)

from utils.seku_crd_corpus_utils import *
from prediction.predict_ner import *
import random

### Corpus Explorartion and Validation from S3 ###

**Check how many papers in S3**

This may take few minutes for the first run

In [3]:
print(AWS_BUCKET_NAME)
print(AWS_DIRECTORY)
stats = count_s3_files(bucket_name=AWS_BUCKET_NAME, directory=AWS_DIRECTORY)
print(stats)

psm-iiser
SEKU-CRD-v1/index_v1

S3 Directory: s3://psm-iiser/SEKU-CRD-v1/index_v1
JSON Files: 617551
Total Size: 105.34 GB
{'total_files': 617551, 'total_size_gb': 105.34}


**Load Main Papers (Paper that refer/cite other papers)**

In [4]:
paper_index = load_paper_index()
all_main_paper_ids = list(paper_index.keys())
nb_main_papers = len(all_main_paper_ids)
print(f'Total {nb_main_papers} Main papers')

Loading paper index: 166310line [00:24, 6895.55line/s] 


Loaded 166310 papers from /Work/IISER/citation-data/SEKU-CRD-v1/paper_index_2024.jsonl (6496.9 MB in memory)
Total 166310 Main papers


**Random Paper Pickup and validation**

In [5]:
rand_idx = random.randint(0,nb_main_papers-1)
rand_paper_id = all_main_paper_ids[rand_idx]
print(f"Random Main paper Index:{rand_idx} [Corpus ID: {rand_paper_id}]")
main_data = paper_index[rand_paper_id] # data from paper index file
title = main_data["title"]
nb_all_refs = len(main_data["bib_entry_ids"]) # [A] all unfiltered original references
nb_usable_refs = len(main_data["references"]) # [B] all references that have either an openalex or arxiv id, but does not ensure in-corpus membership
nb_inc_refs = len(main_data["in_corpus_references"]) # [C] all references that have either an openalex or arxiv id, and in-corpus membership
ratio = nb_inc_refs / nb_all_refs # [D] in corpus / original unfiltered
print(f'Title: {title}')
print(f"[Derived] All Refs [{nb_all_refs}],Usable Refs [{nb_usable_refs}], In-Corpus_Refs [{nb_inc_refs}], Ratio [{ratio:.2f}]")

# some values are precomputed in paper index, they should have same value as computed above
nb_all_refs_r = main_data["total_bib_entries"] # basically len(A)
nb_inc_refs_r = main_data["nb_reference_in_corpus"] # basically len(C)
ratio_r = main_data["reference_ratio"] # should be same as in D
print(f"[Precomputed] All Refs [{nb_all_refs_r}], In-Corpus_Refs [{nb_inc_refs_r}], Ratio [{ratio_r:.2f}]")

Random Main paper Index:123957 [Corpus ID: 1510.00494]
Title: Study of vacuum behavior for inert models with discrete $Z_{2}$-like and
  abelian $U(1)$ symmetries
[Derived] All Refs [83],Usable Refs [43], In-Corpus_Refs [27], Ratio [0.33]
[Precomputed] All Refs [83], In-Corpus_Refs [27], Ratio [0.33]


**Checking Paper Content Main and Refs [only abstract] from S3**

In [6]:
in_corpus_refs_ids = main_data["in_corpus_references"] # list of IDs of [C]
# we need to check if these ids truly exists in our Full Paper Data
# our full paper data is in S3 😊, so we will get it from there. 
results = await get_papers_from_s3(in_corpus_refs_ids)
found_full_data = 0
main_paper_data = get_paper_from_s3(rand_paper_id)
main_title,main_abstract = get_title_and_abstract(main_paper_data)
print(f"Checking refernces for Main paper {rand_paper_id}\n{main_title}")
print("="*5)
for paper_id, data in results.items():
    if data is not None:
        ref_title,ref_abstract = get_title_and_abstract(data)
        print(f"{paper_id} found 😁 \n {ref_title}")
        found_full_data += 1
    else:
        print(f"{paper_id} Not found ☹️")
full_data_ratio = found_full_data / nb_all_refs_r
print(f"{full_data_ratio*100:0.2f}% references have full data")

Fetching papers: 100%|██████████| 27/27 [00:01<00:00, 21.70it/s]


Checking refernces for Main paper 1510.00494
Study of vacuum behavior for inert models with discrete $Z_{2}$-like and
  abelian $U(1)$ symmetries
=====
hep-ph/9903289 found 😁 
 Mass bounds of the lightest CP-even Higgs boson in the two-Higgs-doublet
  model
hep-ph/9811234 found 😁 
 Vacuum Stability Bounds in the Two-Higgs Doublet Model
hep-ph/0201160 found 😁 
 Effective operators and vacuum instability as heralds of new physics
1307.5193 found 😁 
 Stability, Higgs Boson Mass and New Physics
hep-ph/0104016 found 😁 
 On the metastability of the Standard Model vacuum
1511.02879 found 😁 
 One-loop contributions to neutral minima in the inert doublet model
1205.6497 found 😁 
 Higgs mass and vacuum stability in the Standard Model at NNLO
1503.07078 found 😁 
 Inert scalars and vacuum metastability around the electroweak scale
hep-ph/9506284 found 😁 
 Dynamics of Two Higgs Doublet CP Violation and Baryogenesis at the
  Electroweak Phase Transition
1503.02135 found 😁 
 Search for a 'stable alig